# TARP Calibration Analysis — DiffDock

Tests whether DiffDock's pose distribution is calibrated using the TARP method (Lemos & Coogan et al. 2023).

**ECP above diagonal** = over-dispersed (too many far-from-binding-site poses)  
**ECP below diagonal** = mode-collapsed (misses valid binding modes)

In [ ]:
import sys, warnings, os
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from utils.tarp_eval import (
    build_results_index, run_tarp_eval,
    ecp_from_fractions, plot_ecp, atc_score
)

RESULTS_FULL = "results/testset_eval_full"
MERGED       = "results/testset_eval_merged"
DATA_DIR     = "data/PDBBind_processed"

complex_names = np.load(f"{MERGED}/complex_names.npy", allow_pickle=True)
results_index = build_results_index(RESULTS_FULL)
print(f"Complexes: {len(complex_names)}, indexed: {len(results_index)}")

## Phase 1 — Centroid TARP (fast)

Represents each pose as its ligand centroid (3D). Tests whether DiffDock samples the correct region of the protein (binding-site localisation).

In [ ]:
# K=100 reference points per complex, ~5–10 seconds on CPU
f_centroid = run_tarp_eval(
    complex_names, results_index, DATA_DIR,
    K=100, mode="centroid", seed=42, verbose=True
)
np.save(f"{MERGED}/tarp_fractions_centroid.npy", f_centroid)
print(f"Collected {len(f_centroid)} (complex, reference) pairs")

In [ ]:
ecp_c, alpha_c = ecp_from_fractions(f_centroid, n_bins=50)
print(f"ATC score (centroid): {atc_score(ecp_c, alpha_c):.4f}")
print(f"  (positive = over-dispersed, negative = mode-collapsed, 0 = perfect)")

fig, ax = plt.subplots(figsize=(5, 5))
plot_ecp(ecp_c, alpha_c, ax=ax, label="DiffDock-L (centroid)", color="C0")
ax.set_title("TARP — Centroid (binding-site localisation)")
plt.tight_layout()
plt.savefig(f"{MERGED}/tarp_ecp_centroid.png", dpi=150, bbox_inches='tight')
plt.show()

## Phase 2 — Full RMSD TARP

Uses symmetry-corrected RMSD (spyrmsd) over all heavy atoms. Tests full 3D pose calibration including orientation and torsion angles.

> **Runtime:** ~20–40 min for K=100 on CPU. Run the cell below on a compute node, or reduce K for a quick check.

In [ ]:
# Set K=10 for a fast local test; use K=100 for the full result (run on compute node)
K_RMSD = 10  # increase to 100 for publication-quality results

f_rmsd = run_tarp_eval(
    complex_names, results_index, DATA_DIR,
    K=K_RMSD, mode="rmsd", seed=42, verbose=True
)
np.save(f"{MERGED}/tarp_fractions_rmsd_K{K_RMSD}.npy", f_rmsd)
print(f"Collected {len(f_rmsd)} (complex, reference) pairs")

In [ ]:
ecp_r, alpha_r = ecp_from_fractions(f_rmsd, n_bins=50)
print(f"ATC score (RMSD, K={K_RMSD}): {atc_score(ecp_r, alpha_r):.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
plot_ecp(ecp_r, alpha_r, ax=ax, label=f"DiffDock-L (RMSD, K={K_RMSD})", color="C1")
ax.set_title("TARP — Full RMSD (pose calibration)")
plt.tight_layout()
plt.savefig(f"{MERGED}/tarp_ecp_rmsd.png", dpi=150, bbox_inches='tight')
plt.show()

## Combined plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_ecp(ecp_c, alpha_c, ax=axes[0], label="Centroid", color="C0")
axes[0].set_title("Centroid (binding-site localisation)")
plot_ecp(ecp_r, alpha_r, ax=axes[1], label=f"Full RMSD (K={K_RMSD})", color="C1")
axes[1].set_title("Full RMSD (pose calibration)")
plt.suptitle("TARP calibration — DiffDock-L on PDBBind test set (n=322)", y=1.02)
plt.tight_layout()
plt.savefig(f"{MERGED}/tarp_ecp_combined.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"ATC centroid: {atc_score(ecp_c, alpha_c):+.4f}")
print(f"ATC RMSD:     {atc_score(ecp_r, alpha_r):+.4f}")

## Per-complex coverage distribution

For each complex, compute the mean coverage fraction (average f over K references). High mean f = over-dispersed for that complex; low mean f = mode-collapsed.

In [ ]:
# Reshape fractions back to (n_complexes, K)
n_valid = len(f_centroid) // 100  # assumes exactly K=100 per complex succeeded
if len(f_centroid) % 100 == 0:
    f_c_mat = f_centroid.reshape(n_valid, 100)
    mean_f_per_complex = f_c_mat.mean(axis=1)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(mean_f_per_complex, bins=30, color="C0", edgecolor="white", alpha=0.85)
    ax.axvline(0.5, color="k", linestyle="--", label="Perfect calibration (0.5)")
    ax.set_xlabel("Mean coverage fraction per complex")
    ax.set_ylabel("Count")
    ax.set_title("Per-complex TARP coverage (centroid)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{MERGED}/tarp_per_complex_hist.png", dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Complexes with mean f > 0.5 (over-dispersed): {(mean_f_per_complex > 0.5).mean()*100:.1f}%")
    print(f"Complexes with mean f < 0.5 (mode-collapsed): {(mean_f_per_complex < 0.5).mean()*100:.1f}%")